In [109]:
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.multiclass import OneVsOneClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

df = pd.read_csv('../winequality-red-clean.csv', delimiter=',')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1458 entries, 0 to 1457
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1458 non-null   float64
 1   volatile acidity      1458 non-null   float64
 2   citric acid           1458 non-null   float64
 3   residual sugar        1458 non-null   float64
 4   chlorides             1458 non-null   float64
 5   free sulfur dioxide   1458 non-null   float64
 6   total sulfur dioxide  1458 non-null   float64
 7   density               1458 non-null   float64
 8   pH                    1458 non-null   float64
 9   sulphates             1458 non-null   float64
 10  alcohol               1458 non-null   float64
 11  quality               1458 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 136.8 KB


In [110]:

X = df.drop(columns=['quality'])
y = df['quality']

standard_scaler = StandardScaler()
X = standard_scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


### Here, we have the ***train_test_accuracy*** function to print training and testing accuracies, and the ***calculate_bias_variance*** function to determine whether the model is underfitting, overfitting, or balanced by calculating bias and variance.

In [111]:
def train_test_accuracy(train_accuracy, test_accuracy):
    print(f'Training Accuracy: {(train_accuracy * 100).__round__(3)}%')
    print(f'Testing Accuracy: {(test_accuracy * 100).__round__(3)}%')

In [112]:
def calculate_bias_variance(model, X_test, y_test):
    # Predicting using the model
    predictions = model.predict(X_test)

    # Calculating bias
    mean_prediction = np.mean(predictions)
    true_mean = np.mean(y_test)
    bias = mean_prediction - true_mean

    # Calculating variance
    variance = np.mean((predictions - mean_prediction) ** 2)

    # Printing bias and variance
    print(f"Bias: {bias}")
    print(f"Variance: {variance}")

    # Define separate thresholds
    bias_threshold = 0.15
    variance_threshold = 0.2

    if abs(bias) > bias_threshold and variance < variance_threshold:
        print("Model Status: Underfitting - High bias and low variance indicate the model is too simplistic.")
    elif abs(bias) < bias_threshold and variance > variance_threshold:
        print("Model Status: Overfitting - Low bias and high variance indicate the model fits training data too well.")
    elif abs(bias) > bias_threshold and variance > variance_threshold:
        print(
            "Model Status: Unbalanced - Both bias and variance are high, suggesting the model may not generalize well.")
    else:
        print("Model Status: Balanced - The model has a good balance between bias and variance.")

### Support Vector Machine (SVM) is a supervised machine learning model that finds the optimal hyperplane to classify data points by maximizing the margin between classes. It is effective for both linear and non-linear classification problems.

#### ***SVM Linear***: Draws a straight line (or plane) to separate different classes, best if data can be split easily this way. It’s straightforward and works well when the classes are clearly divided.

#### ***SVM RBF***: Uses a more complex curve to separate classes, great for data that isn't easy to split with a straight line. It can handle complicated class shapes by transforming data into a higher space.

#### ***OneVsRest***: Splits the classification problem into multiple "one vs all" tasks—each class is separated from all others. It helps in handling multi-class classification by training one classifier per class.

#### ***OneVsOne***: Creates classifiers for every pair of classes, comparing each class against each other. This approach trains multiple classifiers to determine the best fit for each class pair.

### I will evaluate the following four models: SVM with a Linear kernel using OneVsRest (OVR) and OneVsOne (OVO) strategies, as well as SVM with an RBF kernel using both OVR and OVO approaches. Each model will be tested using all available features to assess training and testing accuracies, as well as bias and variance.

In [113]:
# Linear

ovr_linear = OneVsRestClassifier(SVC(kernel='linear'))
ovr_linear.fit(X_train, y_train)

cv_results = cross_validate(ovr_linear, X, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsRestClassifier LINEAR:')
train_test_accuracy(train_accuracy, test_accuracy)
calculate_bias_variance(ovr_linear, X_test, y_test)

OneVsRestClassifier LINEAR:
Training Accuracy: 56.91%
Testing Accuracy: 54.807%
Bias: -0.20890410958904138
Variance: 0.6463102833552261
Model Status: Unbalanced - Both bias and variance are high, suggesting the model may not generalize well.


In [114]:
#RBF

ovr_rbf = OneVsRestClassifier(SVC(kernel='rbf'))
ovr_rbf.fit(X_train, y_train)

cv_results = cross_validate(ovr_rbf, X, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsRestClassifier RBF:')
train_test_accuracy(train_accuracy, test_accuracy)
calculate_bias_variance(ovr_rbf, X_test, y_test)


OneVsRestClassifier RBF:
Training Accuracy: 71.193%
Testing Accuracy: 58.369%
Bias: -0.09246575342465757
Variance: 0.3659574967160818
Model Status: Overfitting - Low bias and high variance indicate the model fits training data too well.


In [115]:
# Linear

ovo_linear = OneVsOneClassifier(SVC(kernel='linear'))
ovo_linear.fit(X_train, y_train)

cv_results = cross_validate(OneVsOneClassifier(SVC(kernel='linear')), X, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsOneClassifier LINEAR : ')
train_test_accuracy(train_accuracy, test_accuracy)
calculate_bias_variance(ovo_linear, X_test, y_test)

OneVsOneClassifier LINEAR : 
Training Accuracy: 61.557%
Testing Accuracy: 57.751%
Bias: -0.08219178082191814
Variance: 0.3846406455244886
Model Status: Overfitting - Low bias and high variance indicate the model fits training data too well.


In [116]:
# RBF
ovo_rbf = OneVsOneClassifier(SVC(kernel='rbf'))
ovo_rbf.fit(X_train, y_train)

cv_results = cross_validate(ovo_rbf, X, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsOneClassifier RBF:')
train_test_accuracy(train_accuracy, test_accuracy)
calculate_bias_variance(ovo_rbf, X_test, y_test)


OneVsOneClassifier RBF:
Training Accuracy: 69.633%
Testing Accuracy: 58.507%
Bias: -0.08904109589041109
Variance: 0.35851003940701826
Model Status: Overfitting - Low bias and high variance indicate the model fits training data too well.


#### **Notice** : The RBF kernel performs better than the linear kernel, showing higher accuracy and lower bias, while still having some variance. High bias means the model is too simple and can't capture data patterns, leading to underfitting. OneVsOne slightly outperforms OneVsRest, suggesting it generalizes better. Overall, RBF with OneVsOne gives the highest testing accuracy.

### I will now build all four models again using the features selected by the KBest algorithm, which applies the f_classif scoring function. This approach will help determine if feature selection improves model performance. 

In [117]:
'''
Selected Features classification:
 Index(['volatile acidity', 'citric acid', 'total sulfur dioxide', 'sulphates',
       'alcohol'], dtype='object')
'''

selected_features = ['volatile acidity', 'citric acid', 'total sulfur dioxide', 'sulphates',
                     'alcohol']
X_selected = df[selected_features]
y = df['quality']

standard_scaler = StandardScaler()
X_selected = standard_scaler.fit_transform(X_selected)

X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)

In [118]:
# Linear

ovr_linear = OneVsRestClassifier(SVC(kernel='linear'))
ovr_linear.fit(X_train, y_train)

cv_results = cross_validate(ovr_linear, X_selected, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsRestClassifier LINEAR:')
train_test_accuracy(train_accuracy, test_accuracy)
calculate_bias_variance(ovr_linear, X_test, y_test)

OneVsRestClassifier LINEAR:
Training Accuracy: 54.236%
Testing Accuracy: 52.678%
Bias: -0.294520547945206
Variance: 0.5446612872959279
Model Status: Unbalanced - Both bias and variance are high, suggesting the model may not generalize well.


In [119]:
#RBF

ovr_rbf = OneVsRestClassifier(SVC(kernel='rbf'))
ovr_rbf.fit(X_train, y_train)

cv_results = cross_validate(ovr_rbf, X_selected, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsRestClassifier RBF:')
train_test_accuracy(train_accuracy, test_accuracy)
calculate_bias_variance(ovr_rbf, X_test, y_test)

OneVsRestClassifier RBF:
Training Accuracy: 65.998%
Testing Accuracy: 59.671%
Bias: -0.10273972602739789
Variance: 0.3539125539500844
Model Status: Overfitting - Low bias and high variance indicate the model fits training data too well.


In [120]:
# Linear
ovo_linear = OneVsOneClassifier(SVC(kernel='linear'))
ovo_linear.fit(X_train, y_train)

cv_results = cross_validate(ovo_linear, X_selected, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsOneClassifier LINEAR : ')
train_test_accuracy(train_accuracy, test_accuracy)
calculate_bias_variance(ovo_linear, X_test, y_test)

OneVsOneClassifier LINEAR : 
Training Accuracy: 58.848%
Testing Accuracy: 57.618%
Bias: -0.11301369863013733
Variance: 0.321108556952524
Model Status: Overfitting - Low bias and high variance indicate the model fits training data too well.


In [121]:
# RBF
ovo_rbf = OneVsOneClassifier(SVC(kernel='rbf'))
ovo_rbf.fit(X_train, y_train)

cv_results = cross_validate(ovo_rbf, X_selected, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsOneClassifier RBF:')
train_test_accuracy(train_accuracy, test_accuracy)
calculate_bias_variance(ovo_rbf, X_test, y_test)

OneVsOneClassifier RBF:
Training Accuracy: 64.798%
Testing Accuracy: 59.397%
Bias: -0.08561643835616461
Variance: 0.3304911803340214
Model Status: Overfitting - Low bias and high variance indicate the model fits training data too well.


### Feature selection using KBest (f_classif) has led to slight changes in model performance. The RBF models continue to outperform the linear ones, especially in testing accuracy. OneVsOne with RBF achieves the highest testing accuracy, while the overall variance has decreased across models, suggesting improved consistency with selected features.

